In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# ---------- settings ----------
FILE1 = "expert_1.csv"
FILE2 = "expert_2.csv"

OUT = "expert_labeling.csv"
AGREEMENT_OUT = "inter_rater_agreement.csv"

DISAGREE_RULE = "either"  # "either" or "flag"
N_BOOTSTRAP = 2000
RANDOM_SEED = 42

angle_cols = [
    "ELBOW_ANG",
    "SHOULDER_ANG",
    "HIP_ANG",
    "KNEE_ANG",
]
categorical_cols = ["head_side"]
label_cols = categorical_cols + angle_cols
# ------------------------------

# Load and align images
a = pd.read_csv(FILE1).set_index("filename")
b = pd.read_csv(FILE2).set_index("filename")

# Only images present in both files
common = a.index.intersection(b.index)
a, b = a.loc[common].copy(), b.loc[common].copy()

if len(common) == 0:
    raise ValueError("No matching filenames found.")

# Validate required columns
required = ["head_side"] + angle_cols
for col in required:
    if col not in a.columns or col not in b.columns:
        raise ValueError(f"Missing required column: {col}")

# Normalize categorical labels; preserve missing values
a["head_side"] = a["head_side"].astype("string").str.strip().str.upper()
b["head_side"] = b["head_side"].astype("string").str.strip().str.upper()

a["head_side"] = a["head_side"].replace("", pd.NA)
b["head_side"] = b["head_side"].replace("", pd.NA)

# Binary angle flags: blank = 0, as specified
for col in angle_cols:
    a[col] = pd.to_numeric(a[col], errors="raise").fillna(0).astype(int)
    b[col] = pd.to_numeric(b[col], errors="raise").fillna(0).astype(int)

    if not a[col].isin([0, 1]).all() or not b[col].isin([0, 1]).all():
        raise ValueError(f"{col} must contain only 0 and 1.")

# ---------- Inter-rater agreement ----------
rng = np.random.default_rng(RANDOM_SEED)


def agreement_stats(x, y, bootstrap_n=2000):
    """Pairwise-complete agreement statistics for two raters."""
    x = np.asarray(x)
    y = np.asarray(y)

    valid = pd.notna(x) & pd.notna(y)
    x, y = x[valid], y[valid]
    n = len(x)

    if n == 0:
        return {
            "n": 0,
            "percent_agreement": np.nan,
            "cohen_kappa": np.nan,
            "positive_agreement": np.nan,
            "negative_agreement": np.nan,
            "prevalence_expert1": np.nan,
            "prevalence_expert2": np.nan,
            "disagreements": 0,
            "disagreement_rate": np.nan,
            "kappa_ci_lower": np.nan,
            "kappa_ci_upper": np.nan,
            "agreement_ci_lower": np.nan,
            "agreement_ci_upper": np.nan,
        }

    # Works for binary and nominal categorical labels
    same = (x == y)
    p_o = float(np.mean(same))
    kappa = cohen_kappa_score(x, y)

    # For binary variables, label 1 means "error flagged"
    is_binary = set(pd.unique(np.concatenate([x, y]))).issubset({0, 1})

    if is_binary:
        x1, y1 = x == 1, y == 1
        both_positive = np.sum(x1 & y1)
        both_negative = np.sum(~x1 & ~y1)
        only_x = np.sum(x1 & ~y1)
        only_y = np.sum(~x1 & y1)

        positive_den = 2 * both_positive + only_x + only_y
        negative_den = 2 * both_negative + only_x + only_y

        positive_agreement = (
            2 * both_positive / positive_den if positive_den else np.nan
        )
        negative_agreement = (
            2 * both_negative / negative_den if negative_den else np.nan
        )
        prevalence_x = float(np.mean(x1))
        prevalence_y = float(np.mean(y1))
    else:
        positive_agreement = np.nan
        negative_agreement = np.nan
        prevalence_x = np.nan
        prevalence_y = np.nan

    # Percentile bootstrap confidence intervals.
    # Resample image pairs, preserving each pair's two ratings.
    boot_kappa = []
    boot_agreement = []

    if n >= 2 and bootstrap_n > 0:
        for _ in range(bootstrap_n):
            idx = rng.integers(0, n, size=n)
            xb, yb = x[idx], y[idx]

            boot_agreement.append(np.mean(xb == yb))

            # Kappa is undefined for some degenerate resamples
            try:
                kb = cohen_kappa_score(xb, yb)
                if np.isfinite(kb):
                    boot_kappa.append(kb)
            except ValueError:
                pass

    def ci(values):
        if len(values) < 100:
            return np.nan, np.nan
        return tuple(np.percentile(values, [2.5, 97.5]))

    k_lo, k_hi = ci(boot_kappa)
    p_lo, p_hi = ci(boot_agreement)

    return {
        "n": n,
        "percent_agreement": p_o,
        "cohen_kappa": kappa,
        "positive_agreement": positive_agreement,
        "negative_agreement": negative_agreement,
        "prevalence_expert1": prevalence_x,
        "prevalence_expert2": prevalence_y,
        "disagreements": int(np.sum(~same)),
        "disagreement_rate": float(np.mean(~same)),
        "kappa_ci_lower": k_lo,
        "kappa_ci_upper": k_hi,
        "agreement_ci_lower": p_lo,
        "agreement_ci_upper": p_hi,
    }


agreement_rows = []

for col in label_cols:
    stats = agreement_stats(a[col], b[col], N_BOOTSTRAP)
    stats["variable"] = col
    stats["label_type"] = "categorical" if col == "head_side" else "binary"
    agreement_rows.append(stats)

agreement = pd.DataFrame(agreement_rows)
agreement.to_csv(AGREEMENT_OUT, index=False)

# ---------- Create combined labels ----------
res = pd.DataFrame(index=common)
needs_review = pd.Series(False, index=common)

# Head side: retain only when both agree and both are non-missing
hs_valid = a["head_side"].notna() & b["head_side"].notna()
hs_agree = hs_valid & (a["head_side"] == b["head_side"])

res["head_side"] = a["head_side"].where(hs_agree)
needs_review |= ~hs_agree

# Angle flags
for col in angle_cols:
    agree = a[col] == b[col]

    if DISAGREE_RULE == "either":
        res[col] = a[col] | b[col]
    elif DISAGREE_RULE == "flag":
        res[col] = a[col].where(agree)
    else:
        raise ValueError("DISAGREE_RULE must be 'either' or 'flag'.")

    needs_review |= ~agree

res["needs_review"] = needs_review.astype(int)
res.reset_index().to_csv(OUT, index=False)

# ---------- Report ----------
print(f"Matched images: {len(res)}")
print(f"Fully agreed images: {(~needs_review).sum()}")
print(f"Images needing review: {needs_review.sum()}")
print(f"Labels saved to: {OUT}")
print(f"Agreement statistics saved to: {AGREEMENT_OUT}")

display_cols = [
    "variable", "n", "percent_agreement", "cohen_kappa",
    "kappa_ci_lower", "kappa_ci_upper",
    "positive_agreement", "negative_agreement",
    "disagreements", "disagreement_rate",
]
print(agreement[display_cols].round(3).to_string(index=False))


c:\Users\Pashaei\cpu_env\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\Pashaei\cpu_env\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one label in common. `cohen_kappa_score` is undefined and set to the value defined by the the `replace_undefined_by` param, which is set to nan.
  return func(*args, **kwargs)
c:\Users\Pashaei\cpu_env\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\Pashaei\cpu_env\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one 

Matched images: 69
Fully agreed images: 43
Images needing review: 26
Labels saved to: expert_labeling.csv
Agreement statistics saved to: inter_rater_agreement.csv
    variable  n  percent_agreement  cohen_kappa  kappa_ci_lower  kappa_ci_upper  positive_agreement  negative_agreement  disagreements  disagreement_rate
   head_side 69              1.000        1.000           1.000           1.000                 NaN                 NaN              0              0.000
   ELBOW_ANG 69              0.928        0.809           0.625           0.956               0.857               0.951              5              0.072
SHOULDER_ANG 69              0.913        0.527           0.140           0.841               0.571               0.952              6              0.087
     HIP_ANG 69              0.739        0.097          -0.148           0.365               0.250               0.842             18              0.261
    KNEE_ANG 69              0.971        0.817           0.471    

In [3]:
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score

# r1, r2: DataFrames (69 rows) with columns elbow, shoulder, hip, knee; 1 = error, 0 = no error
joints = ['elbow', 'shoulder', 'hip', 'knee']
rng = np.random.default_rng(0)
rows = []
for j in joints:
    a, b = r1[j].values, r2[j].values
    k = cohen_kappa_score(a, b)
    agree = (a == b).mean() * 100
    boots = []
    for _ in range(5000):
        i = rng.integers(0, len(a), len(a))
        if len(set(a[i])) > 1 or len(set(b[i])) > 1:      # skip degenerate resamples
            boots.append(cohen_kappa_score(a[i], b[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    rows.append({'Joint': j, 'Agreement %': round(agree, 1), 'kappa': round(k, 3),
                 '95% CI': f'[{lo:.3f}, {hi:.3f}]'})
print(pd.DataFrame(rows))

NameError: name 'r1' is not defined